In [8]:
# 01_data_profiling.ipynb — Cell 1
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from config import CONFIG

engine = create_engine(CONFIG["db_url"])

def load_raw_table(engine, schema, table):
    query = f'SELECT * FROM "{schema}"."{table}"'
    df    = pd.read_sql(query, engine)
    print(f"Loaded : {df.shape[0]:,} rows x {df.shape[1]} columns")
    print(f"Memory : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
    return df

df = load_raw_table(engine, CONFIG["schema"], CONFIG["raw_table"])

Loaded : 9,031 rows x 13 columns
Memory : 4.2 MB


In [13]:
# Cell 2 — define function
def profile_structure(df):
    profile = pd.DataFrame({
        "dtype"       : df.dtypes,
        "non_null"    : df.notna().sum(),
        "null_count"  : df.isna().sum(),
        "null_pct"    : (df.isna().sum() / len(df) * 100).round(2),
        "unique"      : df.nunique(),
        "sample_value": df.apply(
            lambda col: col.dropna().iloc[0]
            if col.notna().any() else "ALL NULL"
        )
    })
    profile = (profile
               .reset_index()
               .rename(columns={"index": "column"})
               .sort_values("null_pct", ascending=False))
    return profile


In [15]:
# Cell 3 — run
structure_raw = profile_structure(df)
print(structure_raw.to_string(index=False))

     column   dtype  non_null  null_count  null_pct  unique sample_value
    country     str      9031           0       0.0     182  Afghanistan
       iso2     str      9031           0       0.0     182           AF
       iso3     str      9031           0       0.0     182          AFG
iso_numeric   int64      9031           0       0.0     182            4
       year   int64      9031           0       0.0       1         2024
    measure     str      9031           0       0.0       1          inc
       unit     str      9031           0       0.0       1          num
  age_group     str      9031           0       0.0      16         0-14
        sex     str      9031           0       0.0       3            a
risk_factor     str      9031           0       0.0       6          all
       best float64      9031           0       0.0     471      17000.0
         lo float64      9031           0       0.0     406          0.0
         hi float64      9031           0       0.0

In [16]:
# Cell 4 — define
def profile_coverage(df):
    """
    Profile temporal and geographic coverage.
    Critical for a multi-country time series dataset
    — gaps in coverage affect trend analysis.
    """
    print("=== Coverage Profile ===\n")

    # Year range
    print(f"Year range     : {df['year'].min()} to {df['year'].max()}")
    print(f"Years covered  : {df['year'].nunique()}")

    # Country coverage
    print(f"\nCountries      : {df['country'].nunique()}")
    print(f"ISO3 codes     : {df['iso3'].nunique()}")

    # Measures available
    print(f"\nMeasures:")
    print(df["measure"].value_counts().to_string())

    # Age groups available
    print(f"\nAge groups:")
    print(df["age_group"].value_counts().to_string())

    # Sex categories
    print(f"\nSex:")
    print(df["sex"].value_counts().to_string())

    # Risk factors
    print(f"\nRisk factors:")
    print(df["risk_factor"].value_counts(dropna=False).to_string())

    # Rows per country (checks for gaps)
    rows_per_country = df.groupby("iso3").size()
    print(f"\nRows per country: min={rows_per_country.min()} "
          f"max={rows_per_country.max()} "
          f"mean={rows_per_country.mean():.0f}")

    # Countries with fewer rows than expected
    expected_rows = rows_per_country.median()
    sparse_countries = rows_per_country[
        rows_per_country < expected_rows * 0.5
    ]
    if len(sparse_countries) > 0:
        print(f"\nSparse countries (< 50% of median rows):")
        print(sparse_countries.to_string())


In [17]:
# Cell 5 — run
profile_coverage(df)

=== Coverage Profile ===

Year range     : 2024 to 2024
Years covered  : 1

Countries      : 182
ISO3 codes     : 182

Measures:
measure
inc    9031

Age groups:
age_group
all       887
15plus    869
0-14      546
0-4       546
14-Oct    546
15-19     546
15-24     546
20-24     546
25-34     546
35-44     546
45-54     546
14-May    546
9-May     546
55-64     546
65plus    546
18plus    177

Sex:
sex
a    3571
f    2730
m    2730

Risk factors:
risk_factor
all    8190
dia     177
und     177
alc     171
hiv     164
smk     152

Rows per country: min=45 max=50 mean=50


### Data Profiling & Pre-processing Findings

During initial profiling of the 2024 dataset slice, three key structural characteristics were identified:

1. **Excel Date Auto-Formatting Anomalies:** 
   The `age_group` column contains corrupted strings caused by Excel auto-conversions (e.g., `14-Oct`, `14-May`, `9-May`). A mapping dictionary was applied to restore these to valid demographic age ranges (e.g., `10-14`, `5-14`, `5-9`).

2. **Hierarchical Overlaps:**
   * **Sex:** `sex = 'a'` (3,571 rows) represents aggregated totals, while `'f'` and `'m'` (2,730 rows each) represent gender-specific slices.
   * **Risk Factors:** `risk_factor = 'all'` dominates the dataset (8,190 rows). Specific comorbidities (`hiv`, `alc`, `dia`, `smk`, `und`) are only reported for a subset of ~10–12 countries.
   * **Age:** Broad totals (`all`, `15plus`, `18plus`) overlap with 5-year and 10-year age brackets.

3. **Incomplete Country Grids:**
   Row counts vary between 45 and 50 rows per country due to sparse reporting on specific comorbidities.

## Step 4: Estimate Distribution Profile

In [19]:
# Cell 6 — define
def profile_estimates(df, estimate_cols):
    """
    Profile the three estimate columns: best, lo, hi.
    Check for impossible values (lo > best, hi < best).
    """
    stats = df[estimate_cols].agg([
        "count", "min", "max", "mean", "median", "std",
        lambda x: x.quantile(0.25),
        lambda x: x.quantile(0.75),
        "skew"
    ]).T
    stats.columns = [
        "count", "min", "max", "mean", "median",
        "std", "Q1", "Q3", "skew"
    ]
    print("--- Estimate Column Statistics ---")
    print(stats.round(2).to_string())

    # Check for impossible CI values
    impossible_lo = (df["lo"] > df["best"]).sum()
    impossible_hi = (df["hi"] < df["best"]).sum()
    negative_best = (df["best"] < 0).sum()

    print(f"\n--- Data Quality Checks ---")
    print(f"lo > best (impossible): {impossible_lo:,}")
    print(f"hi < best (impossible): {impossible_hi:,}")
    print(f"best < 0  (impossible): {negative_best:,}")

    return stats

# Cell 7 — run
estimate_stats = profile_estimates(
    df, CONFIG["numeric_cols"]
)


--- Estimate Column Statistics ---
       count  min        max      mean  median       std    Q1      Q3   skew
best  9031.0  0.0  2710000.0   7944.20   260.0  60006.34  24.0  1700.0  25.25
lo    9031.0  0.0  2290000.0   3682.86    13.0  45268.26   0.0   180.0  32.01
hi    9031.0  0.0  3120000.0  13911.84   490.0  77973.06  40.0  4000.0  19.72

--- Data Quality Checks ---
lo > best (impossible): 0
hi < best (impossible): 0
best < 0  (impossible): 0


## Estimate Distribution & Data Integrity Assessment

Statistical profiling was conducted on the core metric columns (`best`, `lo`, `hi`) to evaluate data range, skewness, and logical consistency.

### Summary Statistics Table

| Metric | Count | Min | 25% (Q1) | Median | 75% (Q3) | Max | Mean | Std Dev | Skewness |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **`best`** | 9,031 | 0.0 | 24.0 | 260.0 | 1,700.0 | 2,710,000.0 | 7,944.20 | 60,006.34 | **25.25** |
| **`lo`** | 9,031 | 0.0 | 0.0 | 13.0 | 180.0 | 2,290,000.0 | 3,682.86 | 45,268.26 | **32.01** |
| **`hi`** | 9,031 | 0.0 | 40.0 | 490.0 | 4,000.0 | 3,120,000.0 | 13,911.84 | 77,973.06 | **19.72** |

---

### Key Analytical Takeaways & Modeling Implications

1. **High Bounding Consistency:** 
   * **0** rule violations detected ($lo \le best \le hi$ holds across 100% of rows).
   * **0** negative case estimations recorded.

2. **Severe Right-Skewness (Extreme Outliers):**
   * The distribution of incidence estimates is heavily skewed to the right ($\text{Skewness} = 25.25$).
   * **Median ($260$) vs. Mean ($7,944.20$):** Due to high-burden nations and summary total rows, parametric metrics like Mean and Standard Deviation are heavily distorted. 
   * **Action for EDA:** Non-parametric statistics (Medians, IQRs, log-transformations, or top-N rankings) must be used when summarizing distributions across countries.

3. **Uncertainty Interval Spread:**
   * The average upper estimate (`hi` mean: $13,911$) is roughly **3.78x** the average lower estimate (`lo` mean: $3,682$), reflecting the substantial estimation variance in epidemiological modeling across different regions.

## Step 5: Cardinality Profile

In [20]:
# Cell 8 — define (same function as hospital readmission)
def profile_cardinality(df, cols, top_n=15):
    results = []
    for col in cols:
        vc = df[col].value_counts(dropna=False).head(top_n)
        print(f"\n{'='*55}")
        print(f"  {col}  |  Unique: {df[col].nunique()}")
        print(vc.to_string())
        for val, cnt in vc.items():
            results.append({
                "column": col,
                "value" : str(val),
                "count" : cnt,
                "pct"   : round(cnt / len(df) * 100, 2)
            })
    return pd.DataFrame(results)

# Cell 9 — run
cardinality_report = profile_cardinality(
    df, CONFIG["categorical_cols"]
)


  country  |  Unique: 182
country
Afghanistan    50
Albania        50
Algeria        50
Argentina      50
Armenia        50
Australia      50
Austria        50
Azerbaijan     50
Bahamas        50
Bangladesh     50
Belarus        50
Belgium        50
Belize         50
Benin          50
Bhutan         50

  measure  |  Unique: 1
measure
inc    9031

  unit  |  Unique: 1
unit
num    9031

  age_group  |  Unique: 16
age_group
all       887
15plus    869
0-14      546
0-4       546
14-Oct    546
15-19     546
15-24     546
20-24     546
25-34     546
35-44     546
45-54     546
14-May    546
9-May     546
55-64     546
65plus    546

  sex  |  Unique: 3
sex
a    3571
f    2730
m    2730

  risk_factor  |  Unique: 6
risk_factor
all    8190
dia     177
und     177
alc     171
hiv     164
smk     152


## Categorical Cardinality & Dimensional Profile

A cardinality analysis was performed to evaluate the uniqueness, density, and structure of categorical variables within the dataset.

### Summary Table

| Field Name | Unique Values (Cardinality) | Primary Categories & Row Distribution | Dimensional Type |
| :--- | :---: | :--- | :--- |
| **`country`** | 182 | Uniformly distributed (~50 rows per country across 182 nations). | Spatial (Categorical) |
| **`measure`** | 1 | `inc` (100% / 9,031 rows) | Constant Filter |
| **`unit`** | 1 | `num` (100% / 9,031 rows) | Constant Unit |
| **`age_group`** | 16 | Granular brackets (546 rows each) + Summary bands (`all`: 887, `15plus`: 869) | Demographic Hierarchy |
| **`sex`** | 3 | `a` (3,571), `f` (2,730), `m` (2,730) | Demographic Disaggregation |
| **`risk_factor`**| 6 | `all` (8,190), `dia` (177), `und` (177), `alc` (171), `hiv` (164), `smk` (152) | Comorbidity Subsets |

---

### Methodological Takeaways for Analytical Design

1. **Dimensional Constants:** `measure` and `unit` have a cardinality of 1. They can be safely dropped from group-by operations during EDA without losing information.
2. **Symmetrical Gender Slices:** The exact equality between male (`m`: 2,730) and female (`f`: 2,730) observations ensures perfectly balanced side-by-side comparative analyses across all covered age brackets.
3. **Comorbidity Sparsity:** Specific risk factors account for less than 10% of total rows. Comparative risk factor analysis must be restricted to the specific country subset where these comorbidities are actively reported, rather than treated as a globally uniform matrix.

## Step 6: Measure Distribution Profile

In [22]:
# Cell 10
def profile_measure_distribution(df):
    """
    Show row counts per measure-unit combination.
    Critical because mixing measures in analysis
    produces meaningless results.
    """
    measure_unit = (df.groupby(["measure", "unit"])
                     .size()
                     .reset_index(name="row_count")
                     .sort_values("row_count", ascending=False))
    print("--- Rows per Measure-Unit Combination ---")
    print(measure_unit.to_string(index=False))

    # Show year range per measure
    print("\n--- Year Range per Measure ---")
    measure_years = df.groupby("measure")["year"].agg(["min", "max", "nunique"])
    print(measure_years.to_string())

    return measure_unit

measure_dist = profile_measure_distribution(df)

--- Rows per Measure-Unit Combination ---
measure unit  row_count
    inc  num       9031

--- Year Range per Measure ---
          min   max  nunique
measure                     
inc      2024  2024        1


## Measure & Temporal Scope Profile

A profile of the measure and temporal dimensions was conducted to define the boundary conditions of the dataset.

### Summary Table

| Dimension | Characteristic | Observed Value | Analytical Implication |
| :--- | :--- | :---: | :--- |
| **Epidemiological Metric** | `measure` | `inc` (Incidence) | Focuses exclusively on newly diagnosed or estimated TB cases in 2024. |
| **Unit of Measurement** | `unit` | `num` (Absolute Count) | Absolute case volume; raw counts reflect total national disease burden. |
| **Temporal Coverage** | `year` | 2024 (1 unique year) | Pure cross-sectional dataset (no longitudinal time-series modeling). |

---

### Key Methodological Takeaways

1. **Pure Cross-Sectional Analysis:** Because `year` is invariant (2024), all downstream visualizations and statistical modeling will evaluate cross-country and demographic distributions within this single anchor year.
2. **Standardization Requirement:** Because cases are reported as raw counts (`unit = 'num'`), high absolute numbers in populous nations (e.g., India) reflect both demographic scale and disease burden. Normalization by population size is recommended when evaluating relative national risk.